In [1]:
import pandas as pd

# ==========================================
# LOAD WEATHER DATA
# ==========================================

weather_df = pd.read_csv(
    "../datasets/weather/multi_state_weather.csv"
)

# ==========================================
# LOAD NDVI DATA
# ==========================================

ndvi_df = pd.read_csv(
    "../datasets/satellite/multi_state_ndvi.csv"
)

# ==========================================
# LOAD YIELD DATA
# ==========================================

yield_df = pd.read_csv(
    "../datasets/yield/multi_state_all_crops_yield.csv"
)

# ==========================================
# CLEAN COLUMN NAMES
# ==========================================

weather_df.columns = weather_df.columns.str.strip()
ndvi_df.columns = ndvi_df.columns.str.strip()
yield_df.columns = yield_df.columns.str.strip()

# ==========================================
# CLEAN STRINGS
# ==========================================

for df in [weather_df, ndvi_df, yield_df]:

    df['State_Name'] = (
        df['State_Name']
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df['District_Name'] = (
        df['District_Name']
        .astype(str)
        .str.strip()
        .str.lower()
    )

# ==========================================
# CLEAN CROP COLUMN
# ==========================================

yield_df['Crop'] = (

    yield_df['Crop']

    .astype(str)

    .str.strip()

    .str.lower()

)

# ==========================================
# FIX YEAR TYPES
# ==========================================

weather_df['Year'] = weather_df['Year'].astype(int)

ndvi_df['Year'] = (
    ndvi_df['Year']
    .astype(float)
    .astype(int)
)

yield_df['Year'] = yield_df['Year'].astype(int)

# ==========================================
# REMOVE UNUSED NDVI COLUMNS
# ==========================================

drop_cols = []

for col in ndvi_df.columns:

    if (
        'system' in col.lower()
        or '.geo' in col.lower()
        or 'mean' in col.lower()
    ):
        drop_cols.append(col)

ndvi_df = ndvi_df.drop(
    columns=drop_cols,
    errors='ignore'
)

# ==========================================
# KEEP REQUIRED NDVI COLUMNS
# ==========================================

ndvi_df = ndvi_df[[
    'State_Name',
    'District_Name',
    'Year',
    'NDVI'
]]

# ==========================================
# PRINT INFO
# ==========================================

print("WEATHER:", len(weather_df))
print("NDVI:", len(ndvi_df))
print("YIELD:", len(yield_df))

# ==========================================
# MERGE WEATHER + NDVI
# ==========================================

merged_df = pd.merge(

    weather_df,

    ndvi_df,

    on=[
        'State_Name',
        'District_Name',
        'Year'
    ],

    how='inner'

)

print("\nAFTER WEATHER + NDVI:")
print(len(merged_df))

# ==========================================
# MERGE WITH YIELD
# ==========================================

final_df = pd.merge(

    merged_df,

    yield_df,

    on=[
        'State_Name',
        'District_Name',
        'Year'
    ],

    how='inner'

)

print("\nFINAL ROWS:")
print(len(final_df))

# ==========================================
# REMOVE NULLS
# ==========================================

final_df = final_df.dropna()

# ==========================================
# VALID NDVI ONLY
# ==========================================

final_df = final_df[
    final_df['NDVI'] > 0
]

# ==========================================
# SAVE
# ==========================================

final_df.to_csv(

    "../datasets/multi_state_final_dataset.csv",

    index=False

)

print("\nMULTI-STATE FINAL DATASET SAVED!")

print(final_df.head())

print("\nFINAL SHAPE:")
print(final_df.shape)

WEATHER: 1080
NDVI: 1020
YIELD: 24929

AFTER WEATHER + NDVI:
890

FINAL ROWS:
20611

MULTI-STATE FINAL DATASET SAVED!
      State_Name District_Name  Year        T2M       RH2M  PRECTOTCORR  \
0  uttar pradesh          agra  2015  26.391288  43.295863       589.35   
1  uttar pradesh          agra  2015  26.391288  43.295863       589.35   
2  uttar pradesh          agra  2015  26.391288  43.295863       589.35   
3  uttar pradesh          agra  2015  26.391288  43.295863       589.35   
4  uttar pradesh          agra  2015  26.391288  43.295863       589.35   

       NDVI          Crop     Yield  
0  0.402542         bajra  1.400917  
1  0.402542  cotton(lint)  0.203390  
2  0.402542         jowar  0.943201  
3  0.402542         maize  1.797297  
4  0.402542          rice  2.266405  

FINAL SHAPE:
(20414, 9)
